# Generate leave-one-cluster-out splits

Generate sequence clusters with MMseqs2, structural clusters with Foldseek,
and active-site feature clusters with K-means. Convert each cluster
partition into complete training/test manifests and generate the fixed
stratified five-fold split.

Sequence, structural, and active-site feature clustering use the 87
proteins in `FullDataset.csv`. Structures are read from `Structures/`,
excluding `O22340_C.pdb`.

Install the additional clustering programs before running this notebook:

```bash
conda activate mtsenv
conda install -c conda-forge -c bioconda mmseqs2 foldseek
```

Generated files are written to `generated_splits/`; the repository's
published split manifests are not overwritten.


## Configure clustering and input files


In [ ]:
import os
from pathlib import Path
import re
import shutil
import subprocess

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler


def repository_root(start):
    for directory in (start, *start.parents):
        if (directory / "FullDataset.csv").is_file():
            return directory
    raise FileNotFoundError("Run this notebook from within the ATC repository.")


ROOT = repository_root(Path.cwd().resolve())
LOCO_DIRECTORY = ROOT / "SupplementaryInformation" / "LOCO"
OUTPUT = LOCO_DIRECTORY / "generated_splits"
OUTPUT.mkdir(parents=True, exist_ok=True)

MODELLING_FILE = ROOT / "FullDataset.csv"
FASTA_FILE = ROOT / "CharacterisedSeqRef.fasta"
STRUCTURES = ROOT / "Structures"
IDENTIFIER = "Protein"
TARGET = "Cyclical"
EXCLUDED_STRUCTURE_IDS = {"O22340_C"}
SEED = 42
MAX_CLUSTER_FRACTION = 0.20
SEQUENCE_IDENTITY = 0.50
SEQUENCE_COVERAGE = 0.90
STRUCTURAL_COVERAGE = 0.95
STRUCTURAL_TM_SCORE = 0.90
STRUCTURAL_EVALUE = 1e-5

CLUSTERING_FEATURES = [
    "Drug Score", "Number of alpha spheres", "Mean alpha-sphere radius",
    "Mean alpha-sphere Solvent Acc.", "Hydrophobicity Score", "Polarity Score",
    "Amino Acid based volume Score", "Pocket volume (Monte Carlo)",
    "Pocket volume (convex hull)", "Charge Score",
    "Local hydrophobic density Score", "Number of apolar alpha sphere",
    "Proportion of apolar alpha sphere", "Glycine - G", "Alanine - A",
    "Leucine - L", "Methionine - M", "Phenylalanine - F", "Tryptophan - W",
    "Lysine - K", "Glutamine - Q", "Glutamic Acid - E", "Serine - S",
    "Proline - P", "Valine - V", "Isoleucine - I", "Cysteine - C",
    "Tyrosine - Y", "Histidine - H", "Arginine - R", "Asparagine - N",
    "Aspartic Acid - D", "Threonine - T", "Number of residues",
    "Ionisable groups", "Polar", "Non-polar", "Charged", "Positive charge",
    "Negative charge", "Pos-neg charge ratio", "Uncharged", "Aromatic",
    "Hydrophobic", "Small", "tiny",
]

missing_programs = [
    package
    for executable, package in (("mmseqs", "mmseqs2"), ("foldseek", "foldseek"))
    if shutil.which(executable) is None
]
if missing_programs:
    raise EnvironmentError(
        "Install these programs in the active environment: "
        + ", ".join(missing_programs)
    )

if not FASTA_FILE.is_file():
    raise FileNotFoundError(f"Required FASTA file not found: {FASTA_FILE}")
if not STRUCTURES.is_dir():
    raise FileNotFoundError(f"Structure directory not found: {STRUCTURES}")

print(f"Output directory: {OUTPUT}")


## Match modelling proteins to their sequences and structures


In [ ]:
def base_identifier(identifier):
    value = Path(str(identifier).strip()).name
    value = re.sub(r"\.pdb(?:_[A-Za-z0-9])?$", "", value, flags=re.I)
    if re.search(r"_[CL]_[A-Za-z0-9]$", value):
        value = value.rsplit("_", 1)[0]
    return re.sub(r"_[CL]$", "", value)


def read_fasta(path):
    sequences = {}
    identifier = None
    fragments = []
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if identifier is not None:
                    sequences[identifier] = "".join(fragments)
                identifier = line[1:].split()[0]
                fragments = []
            else:
                fragments.append(line)
    if identifier is not None:
        sequences[identifier] = "".join(fragments)
    return sequences


dataset = pd.read_csv(MODELLING_FILE)
for column in (IDENTIFIER, TARGET):
    if column not in dataset:
        raise ValueError(f"FullDataset.csv is missing {column}.")
dataset[IDENTIFIER] = dataset[IDENTIFIER].astype(str).str.strip()
dataset[TARGET] = pd.to_numeric(dataset[TARGET], errors="raise").astype(int)
dataset["Protein_Base"] = dataset[IDENTIFIER].map(base_identifier)
if dataset[IDENTIFIER].duplicated().any() or dataset["Protein_Base"].duplicated().any():
    raise ValueError("FullDataset.csv must identify each protein exactly once.")

excluded_bases = {base_identifier(identifier) for identifier in EXCLUDED_STRUCTURE_IDS}
included_exclusions = sorted(set(dataset["Protein_Base"]) & excluded_bases)
if included_exclusions:
    raise ValueError(
        f"Excluded structural proteins appear in FullDataset.csv: {included_exclusions}"
    )

sequences = {
    base_identifier(identifier): sequence
    for identifier, sequence in read_fasta(FASTA_FILE).items()
}
structures = {}
for path in STRUCTURES.rglob("*.pdb"):
    base = base_identifier(path.name)
    if base not in excluded_bases:
        structures.setdefault(base, path)

missing_sequences = sorted(set(dataset["Protein_Base"]) - set(sequences))
missing_structures = sorted(set(dataset["Protein_Base"]) - set(structures))
if missing_sequences or missing_structures:
    raise ValueError(
        f"Missing sequences: {missing_sequences}; "
        f"missing structures: {missing_structures}."
    )

prepared_fasta = OUTPUT / "modelling_sequences.fasta"
staged_structures = OUTPUT / "modelling_structures"
staged_structures.mkdir(exist_ok=True)

with prepared_fasta.open("w", encoding="utf-8", newline="\n") as handle:
    for identifier, base in dataset[[IDENTIFIER, "Protein_Base"]].itertuples(index=False):
        handle.write(f">{identifier}\n{sequences[base]}\n")
        destination = staged_structures / f"{identifier}.pdb"
        if destination.exists():
            continue
        try:
            os.link(structures[base], destination)
        except OSError:
            shutil.copy2(structures[base], destination)

print(f"Prepared {len(dataset)} sequences and structures.")
print(f"Excluded structural proteins: {', '.join(sorted(EXCLUDED_STRUCTURE_IDS))}")


## Cluster protein sequences and structures

Sequence clusters require at least 50% sequence identity and 90%
coverage. Structural clusters require 95% coverage, a TM-score of at
least 0.90, and an E-value no greater than 1 × 10⁻⁵.


In [ ]:
def run_program(arguments):
    arguments = [str(argument) for argument in arguments]
    try:
        subprocess.run(
            arguments,
            cwd=OUTPUT,
            check=True,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
        )
    except subprocess.CalledProcessError as error:
        detail = (error.stderr or error.stdout).strip()
        raise RuntimeError(
            f"{' '.join(arguments)} failed:\n{detail[-4000:]}"
        ) from error


run_program([
    "mmseqs", "easy-cluster", prepared_fasta.name, "sequence",
    "mmseqs_tmp", "--min-seq-id", SEQUENCE_IDENTITY,
    "-c", SEQUENCE_COVERAGE, "--cov-mode", "0",
])
sequence_output = OUTPUT / "sequence_cluster.tsv"
if not sequence_output.is_file():
    raise FileNotFoundError(f"Missing MMseqs2 cluster assignments: {sequence_output}")

run_program(["foldseek", "createdb", staged_structures.name, "structure_db"])
run_program([
    "foldseek", "cluster", "structure_db", "structure_cluster_db",
    "foldseek_tmp", "-c", STRUCTURAL_COVERAGE, "--cov-mode", "0",
    "-e", STRUCTURAL_EVALUE, "--tmscore-threshold", STRUCTURAL_TM_SCORE,
    "--cluster-mode", "0", "--single-step-clustering", "1",
])
run_program([
    "foldseek", "createtsv", "structure_db", "structure_db",
    "structure_cluster_db", "structure_clusters.tsv",
])
structure_output = OUTPUT / "structure_clusters.tsv"
if not structure_output.is_file():
    raise FileNotFoundError(f"Missing Foldseek cluster assignments: {structure_output}")


def read_clusters(path, group_name):
    frame = pd.read_csv(
        path, sep="\t", header=None, names=[group_name, "Member"], dtype=str
    )
    frame["Protein_Base"] = frame["Member"].map(base_identifier)
    frame[group_name] = frame[group_name].map(base_identifier)
    frame = frame[frame["Protein_Base"].isin(dataset["Protein_Base"])]
    if frame["Protein_Base"].duplicated().any():
        raise ValueError(f"{path.name} assigns a protein more than once.")
    return frame[["Protein_Base", group_name]]


clustered = dataset.merge(
    read_clusters(sequence_output, "Sequence_Group"),
    on="Protein_Base", how="left", validate="one_to_one",
).merge(
    read_clusters(structure_output, "Structure_Group"),
    on="Protein_Base", how="left", validate="one_to_one",
)


## Generate active-site feature clusters

Standardize all 46 active-site descriptors in `FullDataset.csv` and
increase the K-means cluster count until no cluster contains more than
20% of the modelling dataset.


In [ ]:
required_columns = {IDENTIFIER, TARGET, *CLUSTERING_FEATURES}
missing_columns = sorted(required_columns - set(dataset.columns))
if missing_columns:
    raise ValueError(f"FullDataset.csv is missing columns: {missing_columns}")

features = dataset[CLUSTERING_FEATURES].apply(pd.to_numeric, errors="raise")
scaled_features = StandardScaler().fit_transform(features.fillna(0))
maximum_size = int(np.floor(len(dataset) * MAX_CLUSTER_FRACTION))

for cluster_count in range(min(5, len(dataset)), len(dataset) + 1):
    labels = KMeans(
        n_clusters=cluster_count,
        random_state=SEED,
        n_init="auto",
    ).fit_predict(scaled_features)
    if np.bincount(labels).max() <= maximum_size:
        break
else:
    raise RuntimeError("No feature partition satisfies the cluster-size limit.")

feature_groups = dataset[["Protein_Base"]].assign(Feature_Group=labels)
clustered = clustered.merge(
    feature_groups,
    on="Protein_Base", how="left", validate="one_to_one",
)

group_columns = ["Sequence_Group", "Structure_Group", "Feature_Group"]
if clustered[group_columns].isna().any().any():
    missing = clustered.loc[
        clustered[group_columns].isna().any(axis=1), IDENTIFIER
    ].tolist()
    raise ValueError(f"Cluster assignments are missing for: {missing}")

print(f"Feature-clustering proteins: {len(dataset)}")
print(f"Evaluation proteins: {len(clustered)}")
for name in group_columns:
    sizes = clustered[name].value_counts()
    print(f"{name}: {len(sizes)} groups; largest group = {sizes.max()}")


## Write LOCO and random-fold manifests

Each manifest includes every training and test protein for each fold.
The random five-fold partition uses stratification and a fixed seed of 42.


In [ ]:
def cluster_folds(group_column):
    ordered_groups = clustered[group_column].value_counts().index
    for index, group in enumerate(ordered_groups):
        in_group = clustered[group_column].eq(group)
        yield {
            "name": f"fold_{index:02d}",
            "cluster": str(group),
            "train": clustered.loc[~in_group, IDENTIFIER].tolist(),
            "test": clustered.loc[in_group, IDENTIFIER].tolist(),
        }


splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
random_folds = [
    {
        "name": f"fold_{index:02d}",
        "cluster": f"fold_{index:02d}",
        "train": clustered.iloc[training][IDENTIFIER].tolist(),
        "test": clustered.iloc[testing][IDENTIFIER].tolist(),
    }
    for index, (training, testing) in enumerate(
        splitter.split(clustered[IDENTIFIER], clustered[TARGET])
    )
]


def save_manifest(filename, folds):
    rows = [
        {
            "Protein_ID": protein,
            "Fold": fold["name"],
            "Split": split.title(),
            "Cluster": fold["cluster"],
        }
        for fold in folds
        for split in ("train", "test")
        for protein in fold[split]
    ]
    manifest = pd.DataFrame(rows)
    held_out = manifest.loc[manifest["Split"].eq("Test"), "Protein_ID"]
    if held_out.nunique() != len(clustered) or len(held_out) != len(clustered):
        raise ValueError(f"{filename} does not withhold every protein once.")
    output = OUTPUT / filename
    manifest.to_csv(output, index=False)
    print(f"{filename}: {manifest['Fold'].nunique()} folds")


save_manifest("splits_seq_loco.csv", list(cluster_folds("Sequence_Group")))
save_manifest("splits_struct_loco.csv", list(cluster_folds("Structure_Group")))
save_manifest("splits_feature_loco.csv", list(cluster_folds("Feature_Group")))
save_manifest("splits_random_k5.csv", random_folds)
